# Outlier Detection — Regression Models

Identifies streets where Linear Regression, XGBoost, and Random Forest predictions are significantly wrong.
A **consensus outlier** is flagged by 2 or more models (|residual| > 3 × std).

Results saved to `../results/`

In [6]:
import pandas as pd
import numpy as np
import geopandas as gpd
import warnings
import folium
warnings.filterwarnings("ignore")
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import xgboost as xgb


In [7]:
# Load dataset
df = pd.read_csv("../../notebooks/_elena/data/bcn_noise_regre_ml_dataset.csv").dropna()
drop_cols = [c for c in ["road_id", "noise_day", "noise_evening", "noise_night"] if c in df.columns]
X_raw = df.drop(columns=drop_cols)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

# Load geometry — construct road_id from OSMnx edge node pairs (u_v_key)
gdf = gpd.read_file("../../notebooks/_elena/data/bcn_osmnx_edges.gpkg")[["u", "v", "key", "geometry"]]
gdf["road_id"] = gdf["u"].astype(str) + "_" + gdf["v"].astype(str) + "_" + gdf["key"].astype(str)
gdf = gdf[["road_id", "geometry"]].to_crs(epsg=4326)
print(f"Dataset: {df.shape[0]} rows | Geometry: {len(gdf)} streets")

Dataset: 12854 rows | Geometry: 16541 streets


## Train models and compute residuals on full dataset

In [8]:
models = {
    "lr":  LinearRegression(),
    "xgb": xgb.XGBRegressor(random_state=42),
    "rf":  RandomForestRegressor(random_state=42, n_jobs=-1),
}

for period in ["day", "evening", "night"]:
    y = df[f"noise_{period}"].values
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

    results = df[["road_id", f"noise_{period}"]].copy().rename(
        columns={f"noise_{period}": "noise_actual"})

    for key, model in models.items():
        model.fit(X_tr, y_tr)
        pred  = model.predict(X)
        resid = y - pred
        results[f"pred_{key}"]  = pred.round(2)
        results[f"resid_{key}"] = resid.round(2)

    # Flag outliers: |resid| > 3 std per model
    for key in models:
        thresh = 3 * results[f"resid_{key}"].std()
        results[f"outlier_{key}"] = (results[f"resid_{key}"].abs() > thresh).astype(int)

    results["consensus"] = results["outlier_lr"] + results["outlier_xgb"] + results["outlier_rf"]

    results.to_csv(f"../results/outliers_regression_{period}.csv", index=False)
    n = (results["consensus"] >= 2).sum()
    print(f"{period}: {n} consensus outliers saved")

day: 128 consensus outliers saved
evening: 126 consensus outliers saved
night: 124 consensus outliers saved


## Generate interactive maps

In [9]:
for period in ["day", "evening", "night"]:
    results = pd.read_csv(f"../results/outliers_regression_{period}.csv")
    merged  = gdf.merge(results, on="road_id", how="inner")
    outliers = merged[merged["consensus"] >= 2]

    m = folium.Map(location=[41.3874, 2.1686], zoom_start=13, tiles="CartoDB positron")

    for _, row in outliers.iterrows():
        resid_avg = (row["resid_lr"] + row["resid_xgb"] + row["resid_rf"]) / 3
        color = "#d73027" if resid_avg > 0 else "#4575b4"
        popup = (f"<b>{row['road_id']}</b><br>"
                 f"Actual: {row['noise_actual']} dB<br>"
                 f"LR: {row['pred_lr']} dB (resid: {row['resid_lr']:+.1f})<br>"
                 f"XGB: {row['pred_xgb']} dB (resid: {row['resid_xgb']:+.1f})<br>"
                 f"RF: {row['pred_rf']} dB (resid: {row['resid_rf']:+.1f})<br>"
                 f"Flagged by: {int(row['outlier_lr'])+int(row['outlier_xgb'])+int(row['outlier_rf'])}/3 models")
        folium.GeoJson(
            row["geometry"].__geo_interface__,
            style_function=lambda f, c=color: {"color": c, "weight": 4, "opacity": 0.9},
            tooltip=f"{row['road_id']} | {row['noise_actual']} dB | avg resid: {resid_avg:+.1f}",
            popup=folium.Popup(popup, max_width=280),
        ).add_to(m)

    m.save(f"../results/map_outliers_{period}.html")
    print(f"{period}: map saved ({len(outliers)} outlier streets)")

day: map saved (128 outlier streets)
evening: map saved (126 outlier streets)
night: map saved (124 outlier streets)


## Export filtered dataset

Remove all streets that are consensus outliers (flagged by ≥ 2 models) in **any** time period, then save the cleaned dataset.

In [10]:
# Union of road_ids flagged as consensus outliers across all three periods
outlier_ids = set()
for period in ["day", "evening", "night"]:
    r = pd.read_csv(f"../results/outliers_regression_{period}.csv")
    outlier_ids.update(r.loc[r["consensus"] >= 2, "road_id"])

# Load source and drop outlier streets
source = pd.read_csv("../../notebooks/_elena/data/bcn_noise_regre_ml_dataset.csv")
filtered = source[~source["road_id"].isin(outlier_ids)].reset_index(drop=True)

out_path = "../../notebooks/_elena/data/bcn_noise_regre_ml_dataset_filtered.csv"
filtered.to_csv(out_path, index=False)
print(f"Consensus outliers removed: {len(outlier_ids)}")
print(f"Rows: {len(source)} → {len(filtered)}")
print(f"Saved to: {out_path}")

Consensus outliers removed: 230
Rows: 12854 → 12624
Saved to: ../../notebooks/_elena/data/bcn_noise_regre_ml_dataset_filtered.csv
